<a href="https://colab.research.google.com/github/broadinstitute/missense-pfes/blob/main/notebooks/G2P_PFES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Protein Feature Enrichment Score (PFES)
This Colab notebook computes PFES for user-specified missense variants in a protein of interest and generate variant-level protein feature summary report.  For detail, please see [GitHub](https://github.com/broadinstitute/missense-pfes) or [preprint](url).

**Required inputs** (see *User Input* section below)
- `gene`: HGNC gene symbol
- `uniprot`: UniProt accession of the canonical isoform
- `variants`: comma-separated missense variants in standard notation (e.g. `S106L, M1V`)"

**Fecth protein feature data**

Visit the [glossry](https://github.com/broadinstitute/missense-pfes/blob/main/glossary_protein_feature.md) to see protein features used to compute PFES. Features are dynamically fetched from [Genomics 2 Proteins portal](https://g2p.broadinstitute.org) via G2P API. See [G2P API client library](https://github.com/broadinstitute/g2papi) or [G2P swagger API](https://g2p.broadinstitute.org/api-docs/) for more details.


# **Step 1: Setup**

In [ ]:
# @title ## Installation/environment setting
# {display-mode: "form"}

import sys, os, subprocess

IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'g2papi', 'kaleido==0.2.1'])
    try:
        import kaleido
        print("✅ Ready — continue running remaining cells.")
    except ImportError:
        print("⚠️ Packages installed. Please restart: Runtime → Restart session, then Run all")
        raise SystemExit()

if IN_COLAB:
    !git clone https://github.com/broadinstitute/missense-pfes.git
    %cd /content/missense-pfes
    sys.path.insert(0, '/content/missense-pfes/src')
    home_dir = '/content/missense-pfes'
else:
    sys.path.insert(0, '../src')
    home_dir = '..'

LOOKUP_PATH = f'{home_dir}/data/pfes_pvalue_lookup.tsv'

# ── Imports
import re, importlib
import numpy as np
import pandas as pd
import requests
from io import StringIO
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

import g2papi

import generate_report as _gr_module
importlib.reload(_gr_module)
from generate_report import generate_report

from pfes_scorer import (MAX_ASA, AA_TABLE, SS9_LABELS, RSA_BINS, RSA_LABELS,
                         PLDDT_BINS, PLDDT_LABELS, PCHEM_CLASSES, FEATURE_COLS,
                         ATTR_PREFIXES, DISTANCE_MATRIX, DOMAIN_FEATURES,
                         FUNCTION_FEATURES, PTM_FEATURES)

# ── Constants
ATTR_ORDER      = list(ATTR_PREFIXES.keys())
AA_ORDER        = list('ACDEFGHIKLMNPQRSTVWY')
Q_THRESHOLD     = 0.01
aa_table        = AA_TABLE
distance_matrix = DISTANCE_MATRIX

g2p_ver = '2026_q1'
print (f"Using G2P release {g2p_ver}")

In [ ]:
# @title ## Load helper functions
# {display-mode: "form"}

# ---------------- Protein feature encoding ----------------
def _is_annotated(val):   return str(val).strip() != '-'
def _combine_glyco(row):
    a, b = str(row.get('O-GalNAc','-')).strip(), str(row.get('O-GlcNAc','-')).strip()
    if a=='-' and b=='-': return '-'
    if a=='-': return b
    if b=='-': return a
    return f'{a},{b}'

def _route_disulfide(val, ds_intra, ds_inter):
    if str(val).strip()=='-': return ds_intra, ds_inter
    parts = str(val).split(':',1)
    if len(parts)==2 and 'interchain' in parts[1].lower():
        ds_inter = val if ds_inter=='-' else f'{ds_inter},{val}'
    else:
        ds_intra = val if ds_intra=='-' else f'{ds_intra},{val}'
    return ds_intra, ds_inter

def _classify_region(val):
    matched = set()
    if str(val).strip()=='-': return matched
    for entry in str(val).split(';'):
        entry = entry.strip()
        if not entry: continue
        if entry=='Disordered':             matched.add('Region/Disordered')
        elif 'interact' in entry.lower():   matched.add('Region/Interaction')
        else:                               matched.add('Region/Others')
    return matched


def _fetch_pi(row):
    merged = ';'.join(v for v in row if str(v).strip() != '-')
    if not merged:
        return [], []

    pairs, sources = [], []
    for pair, source in re.findall(r'([A-Z]\d+-[A-Z]\d+)\s*\(([^)]+)\)', merged):
        pairs.append(pair)
        if source.startswith('PDBs:'):
            sources.extend(x.strip() for x in source.replace('PDBs:', '').split(','))
        elif source.strip().startswith('PAE:'):
            sources.append('AlphaFold2')

    return list(set(pairs)), list(set(sources))

def _fetch_ppi(val):
    if str(val).strip() == '-' or pd.isna(val):
        return [], []

    pairs, sources = [], []
    for pair, source in re.findall(r'([A-Z]:[A-Z]\d+-[A-Z]:[A-Z]\d+)\s*\(([^)]+)\)', val):
        pairs.append(pair)
        if source.startswith('PDBs:'):
            sources.extend(x.strip() for x in source.replace('PDBs:', '').split(','))
        elif source.strip().startswith('PAE:'):
            sources.append('AlphaFold2')

    return list(set(pairs)), list(set(sources))

def annotate_protein_features(pf):
    df = pd.DataFrame(False, index=pf.index, columns=['ResID','RefAA']+FEATURE_COLS)
    df['ResID'] = pf['residueId']
    df['RefAA']  = pf['AA']

    ss_col = 'Secondary structure (DSSP 9-state)*'
    if ss_col in pf.columns:
        def _parse(v):
            if v=='-' or pd.isna(v): return None
            s = str(v).strip(); return s[0] if s[0] in SS9_LABELS else None
        parsed = pf[ss_col].apply(_parse)
        for s in SS9_LABELS: df[f'SS:{s}'] = parsed==s

    asa_col = 'Accessible surface area (Å²)*'
    if asa_col in pf.columns:
        asa  = pd.to_numeric(pf[asa_col], errors='coerce')
        rsa  = (asa / pf['AA'].map(MAX_ASA)).clip(0,1)
        bins = pd.cut(rsa, bins=RSA_BINS, labels=RSA_LABELS, right=False)
        for b in RSA_LABELS: df[f'RSA:{b}'] = bins==b
        df['raw_RSA'] = rsa ### THIS FIX

    plddt_col = 'AlphaFold confidence (pLDDT)'
    if plddt_col in pf.columns:
        plddt = pd.to_numeric(pf[plddt_col], errors='coerce')
        bins  = pd.cut(plddt, bins=PLDDT_BINS, labels=PLDDT_LABELS, right=False)
        for b in PLDDT_LABELS: df[f'pLDDT:{b}'] = bins==b
        df['raw_pLDDT'] = plddt ### THIS FIX

    aa2prp  = aa_table.set_index('one')['prp'].to_dict()
    ref_cls = pf['AA'].map(aa2prp)
    for c in PCHEM_CLASSES: df[f'RefAA:{c}'] = ref_cls==c

    intra_map = {
        'PI:HB_intra': ['Intra-chain Hydrogen bond (PDB)',          'Intra-chain Hydrogen bond (AlphaFold2)'],
        'PI:SB_intra': ['Intra-chain Salt bridge (PDB)',            'Intra-chain Salt bridge (AlphaFold2)'],
        'PI:DS_intra': ['Intra-chain Disulfide bond (PDB)',         'Intra-chain Disulfide bond (AlphaFold2)'],
        'PI:NB_intra': ['Intra-chain Non-bonded interaction (PDB)', 'Intra-chain Non-bonded interaction (AlphaFold2)'],
    }
    for feat, cols in intra_map.items():
        avail = [c for c in cols if c in pf.columns]
        if avail:
            df[feat] = pf[avail].apply(lambda row: any(_is_annotated(v) for v in row), axis=1)

    if 'Disulfide bond' in pf.columns:
        for i, val in enumerate(pf['Disulfide bond']):
            ci = '-' if not df.at[i,'PI:DS_intra']  else 'annotated'
            cx = '-' if not df.at[i,'PPI:DS_inter'] else 'annotated'
            ni, nx = _route_disulfide(val, ci, cx)
            df.at[i,'PI:DS_intra']  = ni!='-'
            df.at[i,'PPI:DS_inter'] = nx!='-'

    inter_map = {
        'PPI:HB_inter':'Inter-chain Hydrogen bond (PDB)',
        'PPI:SB_inter':'Inter-chain Salt bridge (PDB)',
        'PPI:DS_inter':'Inter-chain Disulfide bond (PDB)',
        'PPI:NB_inter':'Inter-chain Non-bonded interaction (PDB)',
    }
    for feat, col in inter_map.items():
        if col in pf.columns: df[feat] = df[feat] | pf[col].apply(_is_annotated)

    for f in FUNCTION_FEATURES: df[f'Function:{f}'] = pf[f].apply(_is_annotated)

    region_cats = pf['Region'].apply(_classify_region)
    for sub in ['Region/Disordered','Region/Interaction','Region/Others']:
        df[f'Domain:{sub}'] = region_cats.apply(lambda s: sub in s)
    for d in [d for d in DOMAIN_FEATURES if not d.startswith('Region/')]:
        df[f'Domain:{d}'] = pf[d].apply(_is_annotated)

    glyco = pf[['O-GalNAc','O-GlcNAc']].apply(_combine_glyco, axis=1)
    df['Modification:O-GalNAc/GlcNAc'] = glyco.apply(_is_annotated)
    for m in [m for m in PTM_FEATURES if m!='O-GalNAc/GlcNAc']:
        df[f'Modification:{m}'] = pf[m].apply(_is_annotated)

    return df

# ---------------- Compute PFES ----------------
def get_or_lookup(odd_ratio_data):
    sig       = odd_ratio_data[odd_ratio_data['q_value'] < Q_THRESHOLD].dropna(subset=['OR'])
    log_ors   = {feat: np.log(row['OR']) for feat, row in sig.iterrows()}
    sig_stats = sig[['OR','q_value']].to_dict(orient='index')
    return log_ors, sig_stats

def add_variant_features(encoded_refaa, alt_aa):
    df      = encoded_refaa.copy()
    aa2prp  = aa_table.set_index('one')['prp'].to_dict()
    alt_cls = aa2prp.get(alt_aa)
    ref_cls = df['RefAA'].map(aa2prp)
    for ref_c in PCHEM_CLASSES:
        for alt_c in PCHEM_CLASSES:
            df[f'AAchange:{ref_c}>{alt_c}'] = (ref_cls==ref_c) & (alt_cls==alt_c)
    dist_bins   = [0, 50, 100, 150, np.inf]
    dist_labels = ['Mild','Moderate','Substantial','Severe']
    dists = df['RefAA'].apply(lambda r: distance_matrix.get(r,{}).get(alt_aa, np.nan))
    grantham_bin = pd.cut(dists, bins=dist_bins, labels=dist_labels, right=False)
    for lbl in dist_labels: df[f'Grantham:{lbl}'] = grantham_bin==lbl
    return df

def compute_pfes_row(feat_row, log_ors):
    scores = {attr: 0.0 for attr in ATTR_PREFIXES}
    for feat, log_or in log_ors.items():
        if feat not in feat_row.index or not feat_row[feat]: continue
        for attr, prefixes in ATTR_PREFIXES.items():
            if any(feat.startswith(p) for p in prefixes):
                scores[attr] += log_or; break
    scores['PFES'] = sum(scores.values())
    # # replace from 0.0 to np.nan if score is 0.0 from scores
    # scores = {k: (np.nan if v == 0.0 else v) for k,v in scores.items()}
    return scores

def parse_variant(v):
    m = re.fullmatch(r'([A-Z])(\d+)([A-Z])', v.strip())
    if not m: raise ValueError(f"Cannot parse variant: {v}")
    return m.group(1), int(m.group(2)), m.group(3)

def build_landscape(encoded_refaa, odd_ratio_data):
    log_ors, sig_stats = get_or_lookup(odd_ratio_data)
    positions = encoded_refaa['ResID'].tolist()
    n         = len(positions)
    keys      = ['PFES'] + list(ATTR_PREFIXES.keys())
    landscape = {k: np.full((n, len(AA_ORDER)), np.nan) for k in keys}
    for j, alt_aa in enumerate(AA_ORDER):
        enc       = add_variant_features(encoded_refaa, alt_aa)
        feat_cols = [c for c in enc.columns if c not in ('ResID','RefAA')]
        for i in range(n):
            if encoded_refaa['RefAA'].iloc[i] == alt_aa: continue
            s = compute_pfes_row(enc.iloc[i][feat_cols], log_ors)
            for k in keys: landscape[k][i,j] = s[k]
    return landscape, sig_stats, positions, log_ors

# ---------------- Heatmap plotting ----------------
ATTR_2D     = {'Physicochemical'}
ATTR_COLORS = {k: 'RdBu_r' for k in ['PFES']+ATTR_ORDER}

def _colorbar_from_domain(fig, row_idx):
    axis_key = 'yaxis' if row_idx==1 else f'yaxis{row_idx}'
    y0, y1   = fig.layout[axis_key].domain
    return (y0+y1)/2, (y1-y0)*1.1

def _sig_features_for_hover(feat_row, log_ors, sig_stats, prefixes):
    lines = []
    for feat, log_or in log_ors.items():
        if not any(feat.startswith(p) for p in prefixes): continue
        if feat not in feat_row.index or not feat_row[feat]: continue
        s = sig_stats.get(feat, {})
        lines.append(f"{feat}  OR={s.get('OR',np.nan):.2f} q={s.get('q_value',np.nan):.2e}")
    return lines if lines else ['(no significant features)']

def build_hover_pfes(landscape, encoded_refaa, log_ors, sig_stats, positions):
    n, ref_aas = len(positions), encoded_refaa['RefAA'].tolist()
    hover = np.empty((n, len(AA_ORDER)), dtype=object)
    for j, alt_aa in enumerate(AA_ORDER):
        enc       = add_variant_features(encoded_refaa, alt_aa)
        feat_cols = [c for c in enc.columns if c not in ('ResID','RefAA')]
        for i in range(n):
            ref, pos = ref_aas[i], positions[i]
            if ref==alt_aa:
                hover[i,j] = f'{ref}{pos}{alt_aa} (synonymous)'; continue
            parts = [f'<b>{ref}{pos}{alt_aa}</b>', f'PFES = {landscape["PFES"][i,j]:.3f}']
            for attr in ATTR_ORDER:
                parts.append(f'{attr} = {landscape[attr][i,j]:.3f}')
            hover[i,j] = '<br>'.join(parts)
    return hover

def build_hover_attr(attr, landscape, encoded_refaa, log_ors, sig_stats, positions):
    prefixes = ATTR_PREFIXES[attr]
    n, ref_aas = len(positions), encoded_refaa['RefAA'].tolist()
    hover = np.empty((n, len(AA_ORDER)), dtype=object)
    if attr in ATTR_2D:
        for j, alt_aa in enumerate(AA_ORDER):
            enc       = add_variant_features(encoded_refaa, alt_aa)
            feat_cols = [c for c in enc.columns if c not in ('ResID','RefAA')]
            for i in range(n):
                ref, pos = ref_aas[i], positions[i]
                if ref==alt_aa:
                    hover[i,j] = f'{ref}{pos}{alt_aa} (synonymous)'; continue
                feats = _sig_features_for_hover(enc.iloc[i][feat_cols], log_ors, sig_stats, prefixes)
                hover[i,j] = f'<b>{ref}{pos}{alt_aa}</b>  score={landscape[attr][i,j]:.3f}<br>'+'<br>'.join(feats)
    else:
        feat_cols = [c for c in encoded_refaa.columns if c not in ('ResID','RefAA')]
        for i in range(n):
            ref, pos  = ref_aas[i], positions[i]
            feats     = _sig_features_for_hover(encoded_refaa.iloc[i][feat_cols], log_ors, sig_stats, prefixes)
            score     = np.nanmean(landscape[attr][i,:])
            cell_text = f'<b>{ref}{pos}</b>  score={score:.3f}<br>'+'<br>'.join(feats)
            for j in range(len(AA_ORDER)): hover[i,j] = cell_text
    return hover

def _add_panel(fig, row_idx, panel, landscape, encoded_refaa, log_ors, sig_stats, positions, x_ticks):
    is_2d  = (panel=='PFES') or (panel in ATTR_2D)
    zdata  = landscape[panel]
    cb_y, cb_len = _colorbar_from_domain(fig, row_idx)
    cb = dict(yref='paper', yanchor='middle', y=cb_y, xanchor='left', x=1.02,
              len=cb_len, thickness=12,
              title=dict(text=panel, side='right', font=dict(size=9)),
              tickfont=dict(size=8))
    if is_2d:
        hover = (build_hover_pfes(landscape, encoded_refaa, log_ors, sig_stats, positions)
                 if panel=='PFES' else
                 build_hover_attr(panel, landscape, encoded_refaa, log_ors, sig_stats, positions))
        trace = go.Heatmap(z=zdata.T, x=x_ticks, y=AA_ORDER, colorscale='RdBu_r', zmid=0,
                           text=hover.T, hovertemplate='%{text}<extra></extra>',
                           colorbar=cb, showscale=True, name=panel)
    else:
        hover = build_hover_attr(panel, landscape, encoded_refaa, log_ors, sig_stats, positions)
        trace = go.Heatmap(z=np.nanmean(zdata, axis=1, keepdims=True).T,
                           x=x_ticks, y=[panel], colorscale='RdBu_r', zmid=0,
                           text=hover[:,0:1].T, hovertemplate='%{text}<extra></extra>',
                           colorbar=cb, showscale=True, name=panel)
    fig.add_trace(trace, row=row_idx, col=1)

    # Fix x and y range
    fig.update_xaxes(range=[-0.5, len(x_ticks) - 0.5], row=row_idx, col=1)
    if is_2d:
        fig.update_yaxes(range=[-0.5, len(AA_ORDER) - 0.5],
                         categoryorder='array', categoryarray=AA_ORDER,
                         row=row_idx, col=1)
    else:
        fig.update_yaxes(range=[-0.5, 0.5],
                         tickvals=[panel], ticktext=[panel],
                         row=row_idx, col=1)


def _apply_layout(fig, gene, variant_str, positions):
    """Apply shared x-axis ticks and layout to any figure."""
    n       = len(positions)
    x_ticks = list(range(n))
    x_labels= [str(p) for p in positions]
    step    = max(1, n // 50)
    fig.update_xaxes(tickvals=x_ticks[::step], ticktext=x_labels[::step],
                     tickfont=dict(size=7), row=7, col=1)
    fig.update_layout(
        title=dict(text=f'{gene} — PFES mutational landscape  ({variant_str})', x=0.5),
        height=1000, width=1300,
        margin=dict(l=60, r=160, t=60, b=50),
        showlegend=False,
    )

def _add_variant_markers(fig, vr, hover_pfes, hover_attrs):
    """Add X markers for one variant onto a figure."""
    attr_partition = vr['interp_df'].set_index('attribute')['partition'].to_dict()
    panel_partition = {
        1: attr_partition.get('Overall',         'PF-Neutral'),
        2: attr_partition.get('Physicochemical', 'PF-Neutral'),
        3: attr_partition.get('Structure',       'PF-Neutral'),
        4: attr_partition.get('Domain',          'PF-Neutral'),
        5: attr_partition.get('Function',        'PF-Neutral'),
        6: attr_partition.get('Modification',    'PF-Neutral'),
        7: attr_partition.get('PPI',             'PF-Neutral'),
    }

    for row_idx in range(1, 8):
        color = PARTITION_COLORS[panel_partition[row_idx]]
        keys  = vr['interp_df'].iloc[row_idx - 1]   # correct row per panel

        if row_idx == 1:
            y_val   = AA_ORDER[vr['yi']]
            hovtext = hover_pfes[vr['xi'], vr['yi']] + f"<br>Partitioning: {keys['partition']}"
        elif row_idx == 2:
            y_val   = AA_ORDER[vr['yi']]
            hovtext = (hover_attrs['Physicochemical'][vr['xi'], vr['yi']] +
                       f"<br>Partitioning: {keys['attribute']} {keys['partition'].replace('PF-', '')}")
        else:
            attr    = ATTR_ORDER[row_idx - 2]
            y_val   = attr
            hovtext = (hover_attrs[attr][vr['xi'], 0] +
                       f"<br>Partitioning: {keys['attribute']} {keys['partition'].replace('PF-', '')}")

        fig.add_trace(
            go.Scatter(
                x=[vr['xi']], y=[y_val],
                mode='markers',
                marker=dict(symbol='x', size=10, color=color,
                            line=dict(width=1, color='black')),
                hovertemplate=hovtext + '<extra></extra>',
                showlegend=False,
            ),
            row=row_idx, col=1,
        )

def heatmap_landscapes(variant_results, landscape, encoded_feature_refaa,
                       log_ors, sig_stats, positions, gene=''):
    import copy
    n       = len(positions)
    x_ticks = list(range(n))

    # Build base figure (no markers)
    fig_base = make_subplots(rows=7, cols=1, shared_xaxes=True,
                             row_heights=[4,4,1,1,1,1,1.2], vertical_spacing=0.05,
                             subplot_titles=['PFES']+ATTR_ORDER)
    for row_idx, panel in enumerate(['PFES']+ATTR_ORDER, start=1):
        _add_panel(fig_base, row_idx, panel, landscape, encoded_feature_refaa,
                   log_ors, sig_stats, positions, x_ticks)

    # Pre-compute hover arrays once
    hover_pfes  = build_hover_pfes(landscape, encoded_feature_refaa,
                                   log_ors, sig_stats, positions)
    hover_attrs = {attr: build_hover_attr(attr, landscape, encoded_feature_refaa,
                                          log_ors, sig_stats, positions)
                   for attr in ATTR_ORDER}

    # Combined figure — all variants marked
    fig_all = copy.deepcopy(fig_base)
    for vr in variant_results:
        _add_variant_markers(fig_all, vr, hover_pfes, hover_attrs)
    variant_str = ', '.join(vr['variant'] for vr in variant_results)
    _apply_layout(fig_all, gene, variant_str, positions)

    # Per-variant figures — one variant marked each
    variant_figs = {}
    for vr in variant_results:
        vfig = copy.deepcopy(fig_base)
        _add_variant_markers(vfig, vr, hover_pfes, hover_attrs)
        _apply_layout(vfig, gene, vr['variant'], positions)
        variant_figs[vr['variant']] = vfig

    fig_all.show()
    return fig_all, variant_figs


# **Step 2: Run PFES pipeline**

In [ ]:
# @title ## Input your gene, protein, variant(s)
# {display-mode: "form"}

# @markdown **Enter either HGNC gene symbol or UniProt accession, or both**. 
gene     = "SOD1"      # @param {type:"string"}
uniprot  = ''            # @param {type:"string"}
## @markdown - Missing identifier will be automatically matched from the [G2P portal](https://g2p-portal.broadinstitute.org/) metadata.

# @markdown **Enter variant(s) in one-leter notation** (e.g. S106L). Multiple variants can be entered as **a comma-separated string**. 

variants = 'S106L'  # @param {type:"string"} 

print(f'----------------------------\n'
      f' User provided input:\n'
      f'  - Gene:     {gene if gene else "(not provided)"}\n'
      f'  - UniProt:  {uniprot if uniprot else "(not provided)"}\n'
      f'  - Variants: {variants}\n'
      f'----------------------------')

print ('\n... Checking the input gene/protin ... ')
# Fetch metadata (used for protein class and gene/uniprot lookup if either is missing)
GCS_META_URL = f'https://storage.googleapis.com/g2p-portal/portal_data/{g2p_ver}_data/uniprot_metadata.tsv'
try:
    r = requests.get(GCS_META_URL); r.raise_for_status()
    meta = pd.read_csv(StringIO(r.text), sep='\t')
except Exception as e:
    raise RuntimeError(f' ❌ Could not fetch metadata: {e}')

# Fill in missing gene or uniprot from metadata if only one is provided
if gene and not uniprot:
    matches = meta.loc[meta['HGNC_symbol'].str.upper() == gene.upper(), 'UniprotKB_Entry'].unique()
    if len(matches) == 0:
        raise ValueError(f' ❌ Gene "{gene}" was not found in the metadata. '
                         f'Please check the gene symbol and try again.')
    elif len(matches) > 1:
        raise ValueError(f'❗ Found {len(matches)} UniProt entries for gene "{gene}": '
                         f'{", ".join(matches)}. '
                         f'Please specify which UniProt accession you want to use.')
    uniprot = matches[0]
    print(f'🔍 UniProt accession for {gene} automatically matched to: {uniprot}')

elif uniprot and not gene:
    matches = meta.loc[meta['UniprotKB_Entry'] == uniprot, 'HGNC_symbol'].unique()
    if len(matches) == 0:
        raise ValueError(f' ❌ UniProt accession "{uniprot}" was not found in the metadata. '
                         f'Please check the accession and try again.')
    elif len(matches) > 1:
        raise ValueError(f'❗ Found {len(matches)} gene symbols for UniProt "{uniprot}": '
                         f'{", ".join(matches)}. '
                         f'Please specify which gene symbol you want to use.')
    gene = matches[0]
    print(f'🔍 Gene symbol for {uniprot} automatically matched to: {gene}')

elif not gene and not uniprot:
    raise ValueError(' ❌ Both gene symbol and UniProt accession are empty. Please provide at least one to proceed.')

print ('\n... Looking up the PANTHER protein class ...')
# Retrieve PANTHER protein class
match = meta.loc[meta['UniprotKB_Entry'] == uniprot, 'PANTHER_protein_class']
protein_class = match.iloc[0] if not match.empty else 'unclassified'
if protein_class == 'not-available':
    protein_class = 'unclassified'

print ('\n... Fetching the protein data ...')

protein_features = g2papi.get_protein_features(gene, uniprot)
if len(protein_features) == 0:
    raise ValueError(f'No protein features found for gene {gene} and UniProt ID {uniprot}')

else:
    float_cols = protein_features.select_dtypes(include='float64').columns
    protein_features[float_cols] = protein_features[float_cols].astype(object)
    protein_features.fillna('-', inplace=True)
    protein_features.rename(columns=lambda c: c.replace(' (UniProt)', ''), inplace=True)

## Process column names
protein_features.rename(columns={'Signal':'Signal peptide'}, inplace=True)
protein_features.rename(columns={'Cross-link':'Crosslinks'}, inplace=True)

# feature encoding
encoded_feature_refaa = annotate_protein_features(protein_features)


# Variant format check and sanity check
print ('\n... Checking the variants ...')

variant_list = [v.strip() for v in variants.split(',')]

valid_format = re.compile(r'^[A-Z]\d+[A-Z]$')

for v in variant_list:
    if not valid_format.match(v):
        raise ValueError(f'❌ Variant "{v}" is not in valid one-letter notation (e.g. C77Y). '
                         f'Please correct your input and try again.')

checked_variants = []
for v in variant_list:
    resid = int(re.findall(r'\d+', v)[0])
    refaa = re.findall(r'[A-Za-z]+', v)[0]

    if resid > len(protein_features):
        raise ValueError(f'❌ Variant "{v}" position {resid} is out of range '
                         f'(protein length: {len(protein_features)}). '
                         f'Please correct your input and try again.')

    if protein_features.iloc[resid - 1].AA != refaa:
        raise ValueError(f'❌ Variant "{v}" reference AA "{refaa}" does not match '
                         f'"{protein_features.iloc[resid - 1].AA}" at position {resid}. '
                         f'Please correct your input and try again.')

    checked_variants.append(v)

variant_list = checked_variants

print(f'\n ✅ Ready to run!'
      f'\n----------------------------',
      f'\n  - Gene:          {gene}'
      f'\n  - UniProt:       {uniprot}'
      f'\n  - Protein class: {protein_class}'
      f'\n  - Variants:      {", ".join(variant_list)}',
      f'\n----------------------------')

In [ ]:
# @title ## Call protein class-specific Odds Ratios (OR) tables
# {display-mode: "form"}

print ("🔍 ... Loading precomputed protein class-specific Odds Ratios (OR) table ... ")
GITHUB_OR_URL = ('https://raw.githubusercontent.com/broadinstitute/missense-pfes'
                 '/refs/heads/main/results/enrichment_OR_by_protein_class.csv')
r = requests.get(GITHUB_OR_URL); r.raise_for_status()
enrichment_df = pd.read_csv(StringIO(r.text), header=[0,1], index_col=0)

if protein_class not in enrichment_df.columns.get_level_values(0):
    print(f"'{protein_class}' not found, falling back to 'unclassified'")
    protein_class = 'unclassified'

odd_ratio_data = enrichment_df[protein_class][['OR','q_value']].copy()
max_val = odd_ratio_data.loc[np.isfinite(odd_ratio_data['OR']), 'OR'].max()
min_val = odd_ratio_data.loc[odd_ratio_data['OR'] > 0, 'OR'].min()
odd_ratio_data['OR'] = np.where(odd_ratio_data['OR']==np.inf, max_val, odd_ratio_data['OR'])
odd_ratio_data['OR'] = np.where(odd_ratio_data['OR']==0,      min_val, odd_ratio_data['OR'])

print (f'\n Effect sizes of protein feature enrichment (OR and q-values): {protein_class}')
odd_ratio_data



In [ ]:
# @title ## Build mutational landscape for a gene/protein
# {display-mode: "form"}
landscape, sig_stats, positions, log_ors = build_landscape(
    encoded_feature_refaa, odd_ratio_data
)
print(f"✅ Landscape built: {len(positions)} positions × {len(AA_ORDER)} alt AAs")


PARTITION_COLORS = {'PF-Enriched':'red', 'PF-Depleted':'blue', 'PF-Neutral':'gray'}
SCORE_COLS       = ['PFES','PFES_Physicochemical','PFES_Structure',
                    'PFES_Domain','PFES_Function','PFES_Modification','PFES_PPI']
PRESENCE_BASED   = {}#'PFES_Function','PFES_PPI'}

LOOKUP_PATH = f'{home_dir}/data/pfes_pvalue_lookup.tsv'
lookup      = pd.read_csv(LOOKUP_PATH, sep='\t')
lookup_idx  = {col: grp.set_index('pfes_value')
               for col, grp in lookup.groupby('score_col')}

def interpret_score(col, score):
    if pd.isna(score): return np.nan, np.nan, 'PF-Neutral'
    if col in PRESENCE_BASED:
        return np.nan, np.nan, 'PF-Enriched' if score>0 else 'PF-Neutral'
    sub = lookup_idx[col]
    idx = sub.index[np.abs(sub.index - score).argmin()]
    p_case, p_control = sub.loc[idx,'p_case'], sub.loc[idx,'p_control']
    if   p_control < 0.05: partition = 'PF-Enriched'
    elif p_case < 0.05: partition = 'PF-Depleted'
    else:              partition = 'PF-Neutral'
    return float(p_case), float(p_control), partition

print(f"\nStart scoring variants ...")
variant_results = []
for v in variant_list:
    try:
        r_aa, v_pos, a_aa = parse_variant(v)
        xi_v = positions.index(v_pos)
        yi_v = AA_ORDER.index(a_aa)

        ref_in_pf = encoded_feature_refaa['RefAA'].iloc[xi_v]
        if ref_in_pf != r_aa:
            print(f"  Warning {v}: expected ref {r_aa}, found {ref_in_pf} — skipping"); continue

        v_scores = {
            'PFES':                landscape['PFES'][xi_v, yi_v],
            'PFES_Physicochemical': landscape['Physicochemical'][xi_v, yi_v],
            'PFES_Structure':       landscape['Structure'][xi_v, yi_v],
            'PFES_Domain':          landscape['Domain'][xi_v, yi_v],
            'PFES_Function':        landscape['Function'][xi_v, yi_v],
            'PFES_Modification':    landscape['Modification'][xi_v, yi_v],
            'PFES_PPI':             landscape['PPI'][xi_v, yi_v],
        }
        interp_rows = []
        for col in SCORE_COLS:
            vscore = v_scores[col]
            vscore = np.nan if vscore == 0 else vscore
            p_case, p_control, part = interpret_score(col, vscore)
            interp_rows.append({
                'attribute':  col.replace('PFES_','') if col!='PFES' else 'Overall',
                'score':      vscore,
                'p_case': p_case, 'p_control': p_control, 'partition': part,
            })
        v_interp_df = pd.DataFrame(interp_rows)


        enc_v     = add_variant_features(encoded_feature_refaa, a_aa).drop(columns=[x for x in encoded_feature_refaa.columns if x.startswith('raw_')])
        feat_cols = [c for c in enc_v.columns if c not in ('ResID','RefAA')]
        feat_row  = enc_v.iloc[xi_v][feat_cols]
        sig_hits  = [
            {'feature': feat, 'log_OR': lo,
                'OR': sig_stats[feat]['OR'], 'q_value': sig_stats[feat]['q_value']}
            for feat, lo in log_ors.items()
            if feat in feat_row.index and feat_row[feat]]
        if len(sig_hits) > 0:
            v_sig_df    = pd.DataFrame(sig_hits).sort_values('log_OR', ascending=False)
        else:
            v_sig_df = pd.DataFrame(columns=['feature','log_OR','OR','q_value'])

        insig_hits = [
            {'feature': feat, 'log_OR': np.log(odd_ratio_data.loc[feat,'OR']),
                'OR':(odd_ratio_data.loc[feat,'OR']),
                'q_value': odd_ratio_data.loc[feat,'q_value']}
            for feat in feat_row[feat_row].index
            if feat not in sig_stats]
        if len(insig_hits) > 0:
            v_insig_df  = pd.DataFrame(insig_hits).sort_values('log_OR', ascending=False)
        else:
            v_insig_df = pd.DataFrame(columns=['feature','log_OR','OR','q_value'])


        overall_part = v_interp_df.loc[v_interp_df['attribute']=='Overall','partition'].iloc[0]
        feat_row = encoded_feature_refaa.iloc[xi_v]
        raw_rows = [ {'feature': 'raw_Grantham', 'value': DISTANCE_MATRIX[r_aa][a_aa]},
            *[{'feature': feat, 'value': feat_row[feat]}
            for feat in feat_row.index if feat.startswith('raw_')]
        ]

        v_raw_df = pd.DataFrame(raw_rows)

        variant_results.append({
            'variant': v, 'ref_aa': r_aa, 'var_pos': v_pos, 'alt_aa': a_aa,
            'xi': xi_v, 'yi': yi_v,
            'interp_df': v_interp_df, 'sig_df': v_sig_df, 'insig_df': v_insig_df, 'raw_df': v_raw_df,
            'partition': overall_part, 'color': PARTITION_COLORS[overall_part],
            'pfes': v_scores['PFES'],
        })
        print(f" - {v}: PFES={v_scores['PFES']:.3f}  [{overall_part}]")
    except Exception as e:
        print(f"  Error {v}: {e}")

print(f"\n✅ {len(variant_results)} variants scored.")


# **Step 3: Visualize and download the output**

In [ ]:
# @title ## Interactive 2D heatmap of mutational landscapes
# @markdown
# {display-mode="form"}

fig, variant_figs = heatmap_landscapes(
    variant_results, landscape, encoded_feature_refaa,
    log_ors, sig_stats, positions, gene=gene,
)

In [ ]:
# @title ## Generate the PFES report and download
# {display-mode="form"}

# @markdown ### Output folder structure
# @markdown After running this cell, outputs are saved to `results/pfes_output/{gene}/`:
# @markdown Click 'Download' to download
# @markdown ```
# @markdown pfes_output_{gene}/
# @markdown ├── {gene}_multi_landscape.html        interactive heatmap, all variants
# @markdown ├── {gene}_multi_landscape.png         static image for figures
# @markdown ├── {gene}_full_landscape.csv          PFES scores for all possible substitutions
# @markdown │
# @markdown └── {variant}/
# @markdown     ├── {gene}_{variant}_landscape.html          interactive heatmap, single variant
# @markdown     ├── {gene}_{variant}_landscape.png           static version
# @markdown     ├── {gene}_{variant}_partitioning.csv        PF partition and scores per attribute
# @markdown     ├── {gene}_{variant}_significant_features.csv   enriched features driving the score
# @markdown     ├── {gene}_{variant}_report.html             Full variant interpretation report
# @markdown ```


import shutil
import ipywidgets as widgets
from google.colab import files

out_dir = f'{home_dir}/results/pfes_output/pfes_output_{gene}'
os.makedirs(out_dir, exist_ok=True)

# ── Shared gene-level outputs ──────────────────────────────────────────────────
fig.write_html(os.path.join(out_dir, f'{gene}_multi_landscape.html'))
fig.write_image(os.path.join(out_dir, f'{gene}_multi_landscape.png'),
                width=1200, height=1000, scale=2)

rows = []
for i, pos in enumerate(positions):
    for j, aa in enumerate(AA_ORDER):
        if np.isnan(landscape['PFES'][i, j]): continue
        row = {'position': pos,
               'ref_aa':   encoded_feature_refaa['RefAA'].iloc[i],
               'alt_aa':   aa,
               'PFES':     landscape['PFES'][i, j]}
        for attr in ATTR_ORDER:
            row[attr] = landscape[attr][i, j]
        rows.append(row)
pd.DataFrame(rows).to_csv(
    os.path.join(out_dir, f'{gene}_full_landscape.csv'), index=False)


# ── Download output folder (Colab only) ───────────────────────────────────────
import shutil
import ipywidgets as widgets

def _download_outputs(b):
    b.description = 'Zipping...'
    b.disabled = True
    zip_path = f'{out_dir}.zip'
    shutil.make_archive(out_dir, 'zip', out_dir)
    files.download(zip_path)
    b.description = '✅ Download started!'

if IN_COLAB:
    from google.colab import files
    btn = widgets.Button(description='📦 Download output folder',
                         button_style='success',
                         layout=widgets.Layout(width='220px', height='35px'))
    btn.on_click(_download_outputs)
    display(btn)

In [ ]:
# @title Run to preview the PFES report for variants
# {display-mode="form"}

for vr in variant_results:
    v       = vr['variant']
    var_dir = os.path.join(out_dir, v)
    os.makedirs(var_dir, exist_ok=True)

    # Retrieve pre-computed variant-specific figure
    vfig     = variant_figs[v]
    png_name = f'{gene}_{v}_landscape.png'

    vfig.write_html(os.path.join(var_dir, f'{gene}_{v}_landscape.html'))
    vfig.write_image(os.path.join(var_dir, png_name), width=1200, height=1000, scale=2)

    vr['sig_df'].to_csv(
        os.path.join(var_dir, f'{gene}_{v}_significant_features.csv'), index=False)
    vr['interp_df'].to_csv(
        os.path.join(var_dir, f'{gene}_{v}_partitioning.csv'), index=False)


    all_sig_insig = pd.concat(
        [vr['sig_df'].dropna(axis=1, how='all'),
        vr['insig_df'].dropna(axis=1, how='all')],
        ignore_index=True
    )

    md_text = generate_report(
        gene               = gene,
        variant            = v,
        protein_class      = protein_class,
        var_pos            = vr['var_pos'],
        ref_aa             = vr['ref_aa'],
        alt_aa             = vr['alt_aa'],
        interp_df          = vr['interp_df'],
        sig_df             = all_sig_insig, #vr['sig_df'],
        insig_df           = vr['insig_df'],
        raw_df             = vr['raw_df'],
        fig                = fig,
        out_dir            = var_dir,
        template_path      = f'{home_dir}/src/templates',
        landscape_png_name = png_name,
    )

    display(Markdown(md_text[:md_text.index("3. PFES Landscape")]))
    print("\n\n")